In [ ]:
import ast
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import root_mean_squared_error, median_absolute_error

In [ ]:
# ----------------------------
# Configuration
# ----------------------------

PATH = "../data/raw/listings.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

# The list of columns to drop is based on a combination of domain knowledge, EDA insights, and practical considerations for modeling. It includes identifiers, URLs, metadata, text fields that are not being processed with NLP techniques, redundant information, and weak predictors. The goal is to simplify the feature space and focus on the most relevant features for predicting the target variable.
COLS_TO_DROP = [
    # identifiers / urls / metadata
    "id", "listing_url", "scrape_id", "source", "picture_url",
    "host_id", "host_url", "host_thumbnail_url", "host_picture_url",
    "calendar_updated", "calendar_last_scraped",

    # leakage
    "estimated_revenue_l365d", "estimated_occupancy_l365d",

    # text fields (no NLP)
    "name", "description", "neighborhood_overview", "host_about",

    # redundant text versions
    "bathrooms_text", "host_name", "host_verifications",

    # review score redundancy
    "review_scores_accuracy", "review_scores_checkin",
    "review_scores_communication", "review_scores_value",

    # availability redundancy
    "availability_60", "availability_eoy",

    # review activity redundancy
    "number_of_reviews_ltm", "number_of_reviews_l30d",
    "number_of_reviews_ly",

    # derived night statistics
    "minimum_minimum_nights", "maximum_minimum_nights",
    "minimum_maximum_nights", "maximum_maximum_nights",
    "minimum_nights_avg_ntm", "maximum_nights_avg_ntm",

    # categorical removal from EDA
    "first_review", "last_review", "license",
    "host_location", "host_neighbourhood", "neighbourhood",
    "neighbourhood_group_cleansed", "has_availability",
    "host_identity_verified",

    # weak categorical predictor
    "host_response_time", "host_response_rate",
]

CLIP_COLS = [
    "beds",
    "bathrooms",
    "bedrooms",
    "minimum_nights",
    "maximum_nights",
    "host_acceptance_rate_num",
]

LOG_COLS = [
    "minimum_nights",
    "accommodates",
    "maximum_nights",
    "number_of_reviews",
    "reviews_per_month",
]

TOP_K = 10
AMENITIES_MIN_SHARE = 0.05

# If you want Ridge to use only engineered distance features instead of raw coordinates,
# leave this as True. Set it to False if you want to keep latitude/longitude too.
DROP_LAT_LON = True

DISTANCE_POINTS = {
    "distance_to_bilbao":   (43.2630, -2.9350),
    "distance_to_donostia": (43.3183, -1.9812),
    "distance_to_vitoria":  (42.8467, -2.6726),
    "distance_to_coast":    (43.3623, -3.0136),
}

In [ ]:
# ----------------------------
# Helper functions
# ----------------------------

def clean_price(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace("", np.nan)
        .astype(float)
    )

def clean_percentage(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace("%", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce") / 100

def parse_amenities(amenity):
    if isinstance(amenity, list):
        parsed = amenity
    elif amenity == "" or amenity is None:
        return []
    else:
        try:
            parsed = ast.literal_eval(amenity)
        except(ValueError, SyntaxError):
            return[]
    return [str(a).strip().strip('"') for a in parsed if str(a).strip()]

def get_frequent_amenities(parsed_amenities, AMENITIES_MIN_SHARE):
    counts = dict()
    n = len(parsed_amenities)
    min_frequency = int(n * AMENITIES_MIN_SHARE)

    for row in parsed_amenities:
        for amenity in set(row):
            counts[amenity] = counts.get(amenity, 0) + 1 # 

    most_frequent = [amenity for amenity in counts if counts[amenity] >= min_frequency]
    
    return most_frequent

def df_encoded_amenities(df, parsed_amenities, most_frequent, original_column="amenities"):
    encoded_rows = []
    for row in parsed_amenities:
        row_set = set(row)
        encoded_row = []
        for amenity in most_frequent:
            if amenity in row_set:
                encoded_row.append(1)
            else:
                encoded_row.append(0)
        encoded_rows.append(encoded_row)

        cleaned_columns = [
        amenity.replace(" ", "_")
               .replace("/", "_or_")
               .replace("-", "_")
               .replace(":", "")
               .replace("’", "")
               .replace("–", "")
        for amenity in most_frequent
    ]
    
    encoded_amenities_df = pd.DataFrame(
        encoded_rows,
        columns=cleaned_columns,
        index=parsed_amenities.index
    )

    df_transformed = df.drop(columns=[original_column]).join(encoded_amenities_df)

    return df_transformed

def distance_to_center(lat, lon):
    center_lat, center_lon = 43.263, -2.935
    distance = np.sqrt((lat - center_lat) ** 2 + (lon - center_lon) ** 2)
    return distance

In [ ]:
# ----------------------------
# Custom transformers
# ----------------------------

class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.columns, errors="ignore")

class NumericLog1pTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.columns:
            X[col] = np.log1p(X[col])
        return X
    
class NumericClipper(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        self.upper_ = {}
        self.lower_ = {}
        for col in self.columns:
            self.upper_[col] = X[col].quantile(0.99)
            self.lower_[col] = X[col].quantile(0.01)
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[col] = X[col].clip(self.lower_, self.upper_)
        return X
    
class PriceToFloat(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = X[self.column].apply(clean_price)
        return X
    
class PercentageCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = X[self.column].apply(clean_percentage)
        return X
    
class AmenitiesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column
    
    def fit(self, X, y=None):
        parsed_amenities = X[self.column].apply(parse_amenities)
        self.most_frequent_ = get_frequent_amenities(parsed_amenities)
        return self

    def transform(self, X):
        X = X.copy()
        parsed_amenities = X[self.column].apply(parse_amenities)
        X_transformed = df_encoded_amenities(X, parsed_amenities, self.most_frequent_, original_column=self.column)
        return X_transformed
    
class HostExperienceTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, host_since_col="host_since", reference_col="last_scraped"):
        self.host_since_col = host_since_col
        self.reference_col = reference_col

    def fit(self, X, y=None):
        self.reference_date_ = pd.to_datetime(
            X[self.reference_col], errors="coerce"
        ).max()
        return self

    def transform(self, X):
        X = X.copy()

        host_since = pd.to_datetime(X[self.host_since_col], errors="coerce")

        X["host_experience_days"] = (
            self.reference_date_ - host_since
        ).dt.days

        return X.drop(columns=[self.host_since_col, self.reference_col], errors="ignore")


class DistanceFeaturesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lat_col="latitude", lon_col="longitude", points=None, drop_lat_lon=True):
        self.points = points
        self.lat_col = lat_col
        self.lon_col = lon_col
        self.drop_lat_lon = drop_lat_lon

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        lat = X[self.lat_col]
        lon = X[self.lon_col]

        for feature_name, (ref_lat, ref_lon) in self.points.items():
            X[feature_name] = np.sqrt((lat - ref_lat) ** 2 + (lon - ref_lon) ** 2)

        if self.drop_lat_lon:
            X = X.drop(columns=[self.lat_col, self.lon_col], errors="ignore")

        return X
    

class BinaryMapTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column, mapping):
        self.column = column
        self.mapping = mapping

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.column in X.columns:
            X[self.column] = X[self.column].map(self.mapping)
        return X


class TopKCategoryTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column, top_k=10, other_label="Other"):
        self.column = column
        self.top_k = top_k
        self.other_label = other_label

    def fit(self, X, y=None):
        counts = X[self.column].value_counts(dropna=True)
        self.top_categories_ = counts.nlargest(self.top_k).index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = X[self.column].where(
            X[self.column].isin(self.top_categories_),
            self.other_label
        )
        return X

In [ ]:
# ----------------------------
# Load data and define target
# ----------------------------

# ------------------------------
# Load data and define target
# ------------------------------

df = pd.read_csv(PATH)

df["price"] = clean_price(df["price"])
df["host_response_rate"] = clean_percentage(df["host_response_rate"])
df["host_acceptance_rate"] = clean_percentage(df["host_acceptance_rate"])
df = df.rename(columns={"host_acceptance_rate": "host_acceptance_rate_num"})

df = df.dropna(subset=["price"]).copy()

df["log_price"] = np.log1p(df["price"])

df["price_bin"] = pd.qcut(df["price"], q=5, duplicates="drop")

df_train, df_test = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["price_bin"]
)

df_train = df_train.drop(columns=["price_bin"]).copy()
df_test = df_test.drop(columns=["price_bin"]).copy()

lower = df_train["log_price"].quantile(0.01)
upper = df_train["log_price"].quantile(0.99)

df_train["log_price"] = df_train["log_price"].clip(lower=lower, upper=upper)
df_test["log_price"] = df_test["log_price"].clip(lower=lower, upper=upper)

X_train = df_train.drop(columns=["price", "log_price"])
y_train = df_train["log_price"]

X_test = df_test.drop(columns=["price", "log_price"])
y_test = df_test["log_price"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4520, 78)
Test shape: (1131, 78)


In [6]:
# ----------------------------
# Shared feature-engineering pipeline
# ----------------------------

feature_engineering = Pipeline(
    steps=[
        ("drop_columns", DropColumnsTransformer(COLS_TO_DROP)),
        ("amenities", AmenitiesTransformer(min_share=AMENITIES_MIN_SHARE)),
        ("host_experience", HostExperienceTransformer()),
        ("distance_features", DistanceFeaturesTransformer(
            points=DISTANCE_POINTS,
            drop_lat_lon=DROP_LAT_LON
        )),
        ("clipper", NumericClipper(CLIP_COLS)),
        ("log1p", NumericLog1pTransformer(LOG_COLS)),
        ("instant_bookable_map", BinaryMapTransformer(
            column="instant_bookable",
            mapping={"t": 1, "f": 0}
        )),
        ("property_topk", TopKCategoryTransformer(
            column="property_type",
            new_column="property_type_clean",
            top_k=TOP_K
        )),
        ("neighbourhood_topk", TopKCategoryTransformer(
            column="neighbourhood_cleansed",
            new_column="neighbourhood_cleansed_clean",
            top_k=TOP_K
        )),
    ]
)

In [7]:
# ----------------------------
# Model-specific preprocessors
# ----------------------------

numeric_selector = make_column_selector(dtype_include=np.number)
categorical_selector = make_column_selector(dtype_exclude=np.number)

ridge_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_selector),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_selector),
    ],
    remainder="drop"
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_selector),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_selector),
    ],
    remainder="drop"
)

In [8]:
# ----------------------------
# Candidate pipelines
# ----------------------------

ridge_pipe = Pipeline([
    ("features", feature_engineering),
    ("preprocess", ridge_preprocessor),
    ("model", Ridge())
])

rf_pipe = Pipeline([
    ("features", feature_engineering),
    ("preprocess", tree_preprocessor),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

hgb_pipe = Pipeline([
    ("features", feature_engineering),
    ("preprocess", tree_preprocessor),
    ("model", HistGradientBoostingRegressor(
        random_state=RANDOM_STATE
    ))
])

In [9]:
# ----------------------------
# Hyperparameter tuning on training data only
# ----------------------------

ridge_grid = {
    "model__alpha": np.logspace(-3, 3, 25),
}

rf_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
}

hgb_grid = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_iter": [200, 300],
    "model__max_depth": [None, 6, 10],
    "model__min_samples_leaf": [10, 20, 30],
    "model__l2_regularization": [0.0, 0.1, 1.0],
}

In [ ]:
ridge_search = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_grid,
    scoring="neg_root_mean_squared_error",
    cv=CV_FOLDS,
    n_jobs=-1,
    refit=True
)

rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=CV_FOLDS,
    n_jobs=-1,
    refit=True,
    random_state=RANDOM_STATE
)

hgb_search = RandomizedSearchCV(
    estimator=hgb_pipe,
    param_distributions=hgb_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=CV_FOLDS,
    n_jobs=-1,
    refit=True,
    random_state=RANDOM_STATE
)

In [11]:
# WARNING:
# Running all three searches can take time.
# You can comment out the tree models temporarily if you want a faster pass.

ridge_search.fit(X_train, y_train)
print("Ridge best CV RMSE:", -ridge_search.best_score_)
print("Ridge best params:", ridge_search.best_params_)

Ridge best CV RMSE: 0.46901844026823375
Ridge best params: {'model__alpha': np.float64(1.7782794100389228)}


In [12]:
rf_search.fit(X_train, y_train)
print("RF best CV RMSE:", -rf_search.best_score_)
print("RF best params:", rf_search.best_params_)

RF best CV RMSE: 0.40481042077620816
RF best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 500}


In [13]:
hgb_search.fit(X_train, y_train)
print("HGB best CV RMSE:", -hgb_search.best_score_)
print("HGB best params:", hgb_search.best_params_)

HGB best CV RMSE: 0.3743005300740429
HGB best params: {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': None, 'model__max_iter': 300, 'model__min_samples_leaf': 10}


In [14]:
# ----------------------------
# Compare tuned models
# ----------------------------

results = pd.DataFrame({
    "model": ["ridge", "random_forest", "hist_gradient_boosting"],
    "cv_rmse_log": [
        -ridge_search.best_score_,
        -rf_search.best_score_,
        -hgb_search.best_score_,
    ]
}).sort_values("cv_rmse_log")

results

,model,cv_rmse_log
2,hist_gradient_boosting,0.374301
1,random_forest,0.404810
0,ridge,0.469018


In [15]:
# ----------------------------
# Select the best model
# ----------------------------

searches = {
    "ridge": ridge_search,
    "random_forest": rf_search,
    "hist_gradient_boosting": hgb_search,
}

best_model_name = results.iloc[0]["model"]
best_search = searches[best_model_name]
best_pipeline = best_search.best_estimator_

print("Selected model:", best_model_name)
print("Best CV RMSE (log scale):", -best_search.best_score_)

Selected model: hist_gradient_boosting
Best CV RMSE (log scale): 0.3743005300740429


In [16]:
# ----------------------------
# Final test evaluation
# ----------------------------

test_pred_log = best_pipeline.predict(X_test)

# Log-scale RMSE
test_rmse_log = root_mean_squared_error(y_test, test_pred_log)

# Convert back to original euro scale
y_test_eur = np.expm1(y_test)
test_pred_eur = np.expm1(test_pred_log)

# Median absolute error in euros
test_medae_eur = median_absolute_error(y_test_eur, test_pred_eur)

print("Final test RMSE (log scale):", test_rmse_log)
print("Final test Median Absolute Error (€):", test_medae_eur)

Final test RMSE (log scale): 0.361078675707378
Final test Median Absolute Error (€): 21.794146142772433


In [17]:
# ==============================
# Actual vs predicted examples
# ==============================

comparison_df = pd.DataFrame({
    "actual_price_eur": y_test_eur,
    "predicted_price_eur": test_pred_eur
}).reset_index(drop=True)

comparison_df["absolute_error_eur"] = (
    comparison_df["actual_price_eur"] - comparison_df["predicted_price_eur"]
).abs()

comparison_df["percentage_error"] = (
    comparison_df["absolute_error_eur"] / comparison_df["actual_price_eur"]
) * 100

print("Random sample of predictions:")
display(comparison_df.sample(10, random_state=42))

print("Worst predictions by absolute error:")
display(comparison_df.sort_values("absolute_error_eur", ascending=False).head(10))

print("Best predictions by absolute error:")
display(comparison_df.sort_values("absolute_error_eur", ascending=True).head(10))

Random sample of predictions:


,actual_price_eur,predicted_price_eur,absolute_error_eur,percentage_error
783,75.0,121.214664,46.214664,61.619551
898,93.0,83.396506,9.603494,10.326337
413,156.0,266.833795,110.833795,71.047304
467,32.0,44.642710,12.642710,39.508468
745,71.0,65.823358,5.176642,7.291046
109,105.0,118.944113,13.944113,13.280108
522,89.0,158.057205,69.057205,77.592365
56,138.0,133.230314,4.769686,3.456294
1111,193.0,187.866452,5.133548,2.659870
816,203.0,225.740937,22.740937,11.202432


Worst predictions by absolute error:


,actual_price_eur,predicted_price_eur,absolute_error_eur,percentage_error
224,9572.0,1161.254967,8410.745033,87.868210
693,9000.0,1066.586714,7933.413286,88.149037
1034,8000.0,1398.633928,6601.366072,82.517076
772,8000.0,2930.568498,5069.431502,63.367894
398,3819.0,555.749551,3263.250449,85.447773
887,3000.0,361.752123,2638.247877,87.941596
1003,2963.0,360.294425,2602.705575,87.840215
400,9286.0,7040.683837,2245.316163,24.179584
971,1954.0,148.732514,1805.267486,92.388305
848,1313.0,291.278335,1021.721665,77.815816


Best predictions by absolute error:


,actual_price_eur,predicted_price_eur,absolute_error_eur,percentage_error
678,103.0,103.035081,0.035081,0.034060
392,66.0,65.933631,0.066369,0.100559
578,105.0,105.077083,0.077083,0.073413
229,254.0,254.149095,0.149095,0.058699
31,120.0,119.847714,0.152286,0.126905
209,86.0,86.188183,0.188183,0.218818
1075,99.0,98.750810,0.249190,0.251707
405,46.0,45.741051,0.258949,0.562933
374,175.0,175.260757,0.260757,0.149004
1062,72.0,71.674746,0.325254,0.451742
